In [1]:
# --------------------------------------------
# a. Read Dataset and create Spark DataFrame -
# --------------------------------------------

spark_df = spark.read.table('FraudDetection.dbo.silver_data_for_MLmodel') #.select(features)


StatementMeta(, dcc6d750-e234-4b7d-b250-2fa061a0f9d4, 3, Finished, Available, Finished, False)

In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Convert Spark → Pandas if needed
df = spark_df.toPandas()

# -------------------------
# Target
# -------------------------
y = df["isFraud"].values

# -------------------------
# Categorical
# -------------------------
cat_cols = ["type", "orig_name_type", "dest_name_type"]

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# -------------------------
# Numeric
# -------------------------
num_cols = [c for c in df.columns if c not in cat_cols + ["isFraud"]]

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# -------------------------
# Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["isFraud"]).values,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

x_train = X_train.astype(np.float32)  # Or np.int64, based on your needs
y_train = y_train.astype(np.int64)
x_test = X_test.astype(np.float32)
y_test = y_test.astype(np.int64)


StatementMeta(, 3a55b07f-7a7b-4ca7-9dcf-685ea0d0dc06, 19, Finished, Available, Finished, False)

In [18]:
import torch
import torch.nn as nn
import torch.utils.data as td
import torch.nn.functional as F
   
# Set random seed for reproducability
torch.manual_seed(0)
   
print("Libraries imported - ready to use PyTorch", torch.__version__)

StatementMeta(, 3a55b07f-7a7b-4ca7-9dcf-685ea0d0dc06, 20, Finished, Available, Finished, False)

Libraries imported - ready to use PyTorch 2.2.1


In [19]:
# Create a dataset and loader for the training data and labels
train_x = torch.Tensor(x_train).float()
train_y = torch.Tensor(y_train).long()
train_ds = td.TensorDataset(train_x,train_y)
train_loader = td.DataLoader(train_ds, batch_size=20,
    shuffle=False, num_workers=1)

# Create a dataset and loader for the test data and labels
test_x = torch.Tensor(x_test).float()
test_y = torch.Tensor(y_test).long()
test_ds = td.TensorDataset(test_x,test_y)
test_loader = td.DataLoader(test_ds, batch_size=20,
                             shuffle=False, num_workers=1)
print('Ready to load data')

StatementMeta(, 3a55b07f-7a7b-4ca7-9dcf-685ea0d0dc06, 21, Finished, Available, Finished, False)

Ready to load data


In [20]:

h1 = 10 

# Define the neural network
class DiabetesNet(nn.Module):
 def __init__(self):
    super(DiabetesNet, self).__init__()
    self.fc1 = nn.Linear(20, h1) # defining the input layer
    self.fc2 = nn.Linear(h1,h1) # defining the hidden layers
    self.fc3 = nn.Linear(h1,2) # defining the output layer

 def forward(self, x):
    fc1_output = torch.relu(self.fc1(x))
    fc2_output = torch.relu(self.fc2(fc1_output))
    y = F.log_softmax(self.fc3(fc2_output).float(), dim=1)
    return y

# Create a model instance from the network
model = DiabetesNet()
print(model)


StatementMeta(, 3a55b07f-7a7b-4ca7-9dcf-685ea0d0dc06, 22, Finished, Available, Finished, False)

DiabetesNet(
  (fc1): Linear(in_features=20, out_features=10, bias=True)
  (fc2): Linear(in_features=10, out_features=10, bias=True)
  (fc3): Linear(in_features=10, out_features=2, bias=True)
)


In [21]:
def train(model, data_loader, optimizer):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    # set the model to training mode
    model.train()
    train_loss=0

    for batch,tensor in enumerate(data_loader):
        data, target = tensor
        optimizer.zero_grad()
        out = model(data)
        loss = loss_criteria(out, target)
        train_loss = train_loss + loss.item()

        # backpropagate adjustments to the weight
        loss.backward()
        optimizer.step()
    
    # Return the average loss 
    avg_loss = train_loss / (batch+1)
    print('Training set: Average loss: {:.6f}'.format(avg_loss))
    return avg_loss

def test(model, data_loader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    # switch the model to evaluation mode
    model.eval()
    test_loss = 0
    correct = 0

    with torch.no_grad():
         batch_count = 0
         for batch, tensor in enumerate(data_loader):
             batch_count += 1
             data, target = tensor
             # get the predictions
             out = model(data)

             # calculate the loss
             test_loss = loss_criteria(out, target).item() + test_loss

             # calculate the accuracy
             _, predicted = torch.max(out.data,1)
             correct += torch.sum(target==predicted).item()

    # Calculate the average loss and total accuracy for this epoch
    avg_loss = test_loss/batch_count
    print('Validation set: Average loss: {:.6f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        avg_loss, correct, len(data_loader.dataset),
        100. * correct / len(data_loader.dataset)))
       
    # return average loss for the epoch
    return avg_loss

StatementMeta(, 3a55b07f-7a7b-4ca7-9dcf-685ea0d0dc06, 23, Finished, Available, Finished, False)

In [23]:
# Specify the loss criteria (we'll use CrossEntropyLoss for multi-class classification)
loss_criteria = nn.CrossEntropyLoss()
   
# Use an optimizer to adjust weights and reduce loss
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
optimizer.zero_grad()
   
# We'll track metrics for each epoch in these arrays
epoch_nums = []
training_loss = []
validation_loss = []
   
# Train over 100 epochs
epochs = 100
for epoch in range(1, epochs + 1):
   
    # print the epoch number
    print('Epoch: {}'.format(epoch))
       
    # Feed training data into the model
    train_loss = train(model, train_loader, optimizer)
       
    # Feed the test data into the model to check its performance
    test_loss = test(model, test_loader)
       
    # Log the metrics for this epoch
    epoch_nums.append(epoch)
    training_loss.append(train_loss)
    validation_loss.append(test_loss)

StatementMeta(, 3a55b07f-7a7b-4ca7-9dcf-685ea0d0dc06, 25, Finished, Available, Finished, False)

Epoch: 1


RuntimeError: mat1 and mat2 shapes cannot be multiplied (20x33 and 20x10)